# Training

Reproduces the MNIST feed-forward experiment from Srivastava et al. (2014),
*Dropout: A Simple Way to Prevent Neural Networks from Overfitting*.

Architecture: 784 -> 1024 (ReLU) -> 1024 (ReLU) -> 10 (softmax),
trained with SGD (lr=0.1, momentum=0.95) for 20 epochs, batch size 128.
We train the network once **without** dropout and once **with** dropout
(p=0.5 hidden), exactly matching the basic Colab implementation.


# Setup (run this first)

1. Edit `REPO_URL` to point at your GitHub repo.
2. Run the cell. It clones the repo into `/content/my-research`, installs
   requirements and adds the repo to `sys.path` so `import src.*` works.

If you do not want to use GitHub yet, upload the `my-research` folder to
Google Drive and instead run:
   from google.colab import drive; drive.mount('/content/drive')
   %cd '/content/drive/MyDrive/my-research'


In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/my-research"  # <-- edit me
%cd /content
!test -d my-research || git clone --depth 1 {REPO_URL} my-research
%cd /content/my-research
!pip install -q -r requirements.txt
import os, sys
sys.path.insert(0, os.getcwd())
print('repo ready at', os.getcwd())


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from src.dataset import mnist_loaders, device
from src.model import Net
from src.utils import plot_histories

torch.manual_seed(42)
device_ = device()
print('Using:', device_)

train_loader, test_loader, _ = mnist_loaders(batch_size=128)
print('train batches:', len(train_loader), '| test images:', len(test_loader.dataset))


In [ ]:
EPOCHS = 20
LR = 0.1
MOMENTUM = 0.95

@torch.no_grad()
def eval_metrics(model, loader, criterion):
    # accuracy and loss measured in eval mode (dropout off)
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device_), labels.to(device_)
        outputs = model(images)
        total_loss += criterion(outputs, labels).item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return total_loss / total, 100.0 * correct / total

def train_one(use_dropout):
    model = Net(dropout=use_dropout).to(device_)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)

    history = {'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': []}
    for epoch in range(EPOCHS):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device_), labels.to(device_)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        t_loss, t_acc = eval_metrics(model, train_loader, criterion)
        e_loss, e_acc = eval_metrics(model, test_loader, criterion)
        history['train_acc'].append(t_acc)
        history['test_acc'].append(e_acc)
        history['train_loss'].append(t_loss)
        history['test_loss'].append(e_loss)
        print(f'Epoch {epoch + 1:2d}/{EPOCHS} |',
              f'Train Acc {t_acc:6.3f}% | Test Acc {e_acc:6.3f}%')
    return model, history


In [ ]:
print("\n=== WITHOUT DROPOUT ===")
model_nodrop, hist_nodrop = train_one(use_dropout=False)

print("\n=== WITH DROPOUT ===")
model_drop, hist_drop = train_one(use_dropout=True)


In [ ]:
# saves results/training_accuracy.png and results/training_loss.png
plot_histories(hist_nodrop, hist_drop, save_dir='results')


In [ ]:
import os
os.makedirs('models', exist_ok=True)
torch.save(model_nodrop.state_dict(), 'models/mnist_nodropout.pt')
torch.save(model_drop.state_dict(), 'models/mnist_dropout.pt')
print('model weights saved to models/')
